# DAAI_N1.4 — Chuyển đổi Silver Data sang 15 bảng chuẩn hóa 3NF

Notebook này đọc các file **Silver** (đã làm sạch) và tái cấu trúc thành các bảng **3NF** theo `Luoc_do_quan_he_3NF.docx`,
mỗi bảng xuất ra **1 file CSV riêng** trong thư mục `OUTPUT_DIR`.

**Trước khi chạy:**
1. Sửa `INPUT_DIR` bên dưới trỏ tới thư mục chứa các file Silver của nhóm.
2. Sửa dict `INPUT_FILES` nếu tên file thật của nhóm khác với tên mặc định đang để.
3. Chạy lần lượt từ trên xuống.

**2 điểm "treo" trong tài liệu 3NF — giờ được TỰ ĐỘNG kiểm tra và tách bằng dữ liệu thật:**
- **Bước 1 (GEOGRAPHY):** kiểm tra `district → city → region` có là phụ thuộc hàm 1-1 không. Nếu đúng, tự tách thành 4 bảng `region`, `city`, `district`, `geography(zip, district_id)`. Nếu không, giữ nguyên 1 bảng `geography` phẳng như bản gốc.
- **Bước 4 (PRODUCT):** kiểm tra `category → segment` có 1-1 không. Nếu đúng, tự tách bảng `category(category_id, segment)` và rút gọn `product`. Nếu không, giữ nguyên `product` như bản gốc.

Cả hai đều **tự quyết định dựa trên dữ liệu thật**, không tách "cứng" theo giả định — kết quả in ra ngay dưới mỗi cell để nhóm biết notebook đã chọn phương án nào.

**Phần cuối (Bước 17)** xây dựng và xuất các bảng **Fact/Dim cho Data Warehouse** phục vụ Đề tài 4: `Dim_Date`, `Dim_Customer`, `Dim_Employee`, `Dim_Product`, `Dim_Geography`, `Fact_Sales`.

## Bước 0: Cấu hình đường dẫn & tiện ích chung

In [2]:
import pandas as pd
import numpy as np
import os
import json

# ==== SỬA 2 DÒNG NÀY CHO ĐÚNG MÁY CỦA BẠN ====
INPUT_DIR = "silver_data"      # thư mục chứa các file *_silver.csv / *_silver.json
OUTPUT_DIR = "warehouse_3nf"   # thư mục sẽ chứa 15 file kết quả
# ================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ánh xạ: tên file silver đầu vào (sửa lại nếu nhóm bạn đặt tên khác)
INPUT_FILES = {
    "geography":            "geography.csv",
    "customer":             "CUSTOMER.csv",
    "sales_employee":       "sales_employee.csv",   # mới: đã có file thật
    "product":              "product.csv",
    "promotion":            "promotion.csv",
    "order":                "ORDER.csv",
    "order_items":          "ORDER_ITEMS.csv",
    "order_item_promotion": "ORDER_ITEM_PROMOTION.csv",  # mới: đã tách sẵn
    "shipper":              "shipper.csv",           # mới: đã tách sẵn
    "shipment":             "shipment.csv",
    "payment":              "payment.csv",
    "returns":              "returns.csv",
    "reviews":              "reviews.csv",
    "inventory":            "inventory.csv",
    "web_traffic":          "web_traffic.csv",
}

def load(key):
    """Đọc 1 file silver theo key trong INPUT_FILES, hỗ trợ cả .csv và .json."""
    fname = INPUT_FILES[key]
    path = os.path.join(INPUT_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Không tìm thấy '{path}'. Kiểm tra lại INPUT_DIR hoặc tên file trong INPUT_FILES['{key}']."
        )
    if path.endswith(".json"):
        return pd.read_json(path)
    return pd.read_csv(path)

def export(df, table_name):
    """Xuất 1 DataFrame thành 1 file CSV riêng, tên đúng theo tên bảng trong lược đồ 3NF."""
    out_path = os.path.join(OUTPUT_DIR, f"{table_name}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Đã xuất {table_name}.csv — {df.shape[0]} dòng, {df.shape[1]} cột -> {out_path}")
    return df


## Bước 1/15: GEOGRAPHY *(zip)*
Kiểm tra phụ thuộc hàm `district → city → region` bằng dữ liệu thật.
- Nếu là quan hệ **1-1** (mỗi district chỉ thuộc đúng 1 city, mỗi city chỉ thuộc đúng 1 region) → tự động tách thành 3 bảng `region`, `city`, `district`, đồng thời rút gọn `geography` chỉ còn `(zip, district_id)` để loại phụ thuộc bắc cầu.
- Nếu **không** phải 1-1 (một district ứng với nhiều city/region khác nhau) → giữ nguyên 1 bảng `geography` phẳng `(zip, city, region, district)` như bản gốc, không ép tách.

Biến `geo_flat` (đầy đủ zip/city/region/district) luôn được giữ lại để các bước sau (SHIPPER) tra cứu, bất kể có tách hay không.

In [3]:
df_geo_raw = load("geography")
geo_flat = df_geo_raw[["zip", "city", "region", "district"]].drop_duplicates(subset="zip").reset_index(drop=True)

# --- Kiểm tra: district -> city -> region có phải phụ thuộc hàm 1-1 không? ---
check_geo = geo_flat.groupby("district")[["city", "region"]].nunique()
n_violation_geo = (check_geo[["city", "region"]] > 1).any(axis=1).sum()
SPLIT_GEOGRAPHY = (n_violation_geo == 0)
print(f"Số district ứng với NHIỀU hơn 1 city/region: {n_violation_geo}")
print("->", "Xác nhận 1-1: TÁCH thành REGION/CITY/DISTRICT/GEOGRAPHY."
      if SPLIT_GEOGRAPHY else "KHÔNG phải 1-1: GIỮ NGUYÊN 1 bảng GEOGRAPHY phẳng.")

if SPLIT_GEOGRAPHY:
    # REGION *(region_id)*
    region = geo_flat[["region"]].drop_duplicates().sort_values("region").reset_index(drop=True)
    region.insert(0, "region_id", region.index + 1)

    # CITY *(city_id)* — mỗi city gắn với đúng 1 region
    city = geo_flat[["city", "region"]].drop_duplicates().sort_values("city").reset_index(drop=True)
    city = city.merge(region, on="region").drop(columns="region")
    city.insert(0, "city_id", city.index + 1)
    city = city[["city_id", "city", "region_id"]]

    # DISTRICT *(district_id)* — mỗi district gắn với đúng 1 city
    district = geo_flat[["district", "city"]].drop_duplicates().sort_values("district").reset_index(drop=True)
    district = district.merge(city[["city_id", "city"]], on="city").drop(columns="city")
    district.insert(0, "district_id", district.index + 1)
    district = district[["district_id", "district", "city_id"]]

    # GEOGRAPHY rút gọn *(zip)* — chỉ còn khóa ngoại district_id
    geography = (geo_flat.merge(district[["district_id", "district"]], on="district")
                          [["zip", "district_id"]]
                          .drop_duplicates(subset="zip")
                          .reset_index(drop=True))

    export(region, "region")
    export(city, "city")
    export(district, "district")
    export(geography, "geography")
else:
    # Fallback: giữ nguyên cấu trúc phẳng như bản gốc, không tách
    geography = geo_flat.copy()
    region = city = district = None
    export(geography, "geography")

geography.head()

Số district ứng với NHIỀU hơn 1 city/region: 39
-> KHÔNG phải 1-1: GIỮ NGUYÊN 1 bảng GEOGRAPHY phẳng.
Đã xuất geography.csv — 39948 dòng, 4 cột -> warehouse_3nf\geography.csv


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


## Bước 2/15: CUSTOMER *(customer_id)*
Bỏ cột `city` (dư thừa vì đã suy ra được qua `zip → GEOGRAPHY.city`), chỉ giữ `zip` làm khóa ngoại.

In [4]:
df_cust_raw = load("customer")

cust_cols = ["customer_id", "zip", "signup_date", "gender", "age_group", "acquisition_channel"]
# Chỉ giữ các cột có tồn tại thật trong file (phòng khi tên cột khác 1 chút)
cust_cols = [c for c in cust_cols if c in df_cust_raw.columns]
customer = df_cust_raw[cust_cols].drop_duplicates(subset="customer_id").reset_index(drop=True)
export(customer, "customer")
customer.head()

Đã xuất customer.csv — 121930 dòng, 6 cột -> warehouse_3nf\customer.csv


,customer_id,zip,signup_date,gender,age_group,acquisition_channel
0,1,15201,2021-12-30,Female,35-44,social_media
1,2,15201,2013-12-27,Female,45-54,email_campaign
2,3,15201,2018-07-24,Female,18-24,organic_search
3,4,15201,2017-11-29,Male,35-44,referral
4,5,15201,2022-09-23,Male,55+,organic_search


## Bước 3/15: SALES_EMPLOYEE *(sales_employee_id)*
⚠️ **Chưa có file dữ liệu gốc cho bảng này** (đã ghi trong Data Dictionary: `sales_employee_id` là FK nhưng
không có bảng định nghĩa nhân viên bán hàng). Cell dưới đây tạo bảng **tạm thời** bằng cách lấy toàn bộ
`sales_employee_id` duy nhất xuất hiện trong ORDER, cột `name` để trống (`NaN`) — **cần nhóm bổ sung dữ liệu
thật (HR data) rồi thay thế file này**, notebook chỉ tạo khung sườn để không bị lỗi FK khi tạo DDL.

In [5]:
df_emp_raw = load("sales_employee")
emp_cols = ["sales_employee_id", "name"]
emp_cols = [c for c in emp_cols if c in df_emp_raw.columns]
sales_employee = df_emp_raw[emp_cols].drop_duplicates(subset="sales_employee_id").reset_index(drop=True)
export(sales_employee, "sales_employee")

# Kiểm tra: có sales_employee_id nào trong ORDER mà không có trong bảng này không?
df_order_check = load("order")
missing_emp = ~df_order_check["sales_employee_id"].isin(sales_employee["sales_employee_id"])
print(f"Số đơn có sales_employee_id KHÔNG khớp SALES_EMPLOYEE: {missing_emp.sum()}")

sales_employee.head()

Đã xuất sales_employee.csv — 200 dòng, 2 cột -> warehouse_3nf\sales_employee.csv
Số đơn có sales_employee_id KHÔNG khớp SALES_EMPLOYEE: 0


,sales_employee_id,name
0,EMP0103,Nguyễn Thị Phúc
1,EMP0180,Lê Gia Hùng
2,EMP0093,Bùi Minh Quân
3,EMP0015,Võ Quốc Mai
4,EMP0107,Hồ Thanh Yến


## Bước 4/15: PRODUCT *(product_id)*
Kiểm tra phụ thuộc hàm `category → segment` bằng dữ liệu thật.
- Nếu mỗi category chỉ ứng với đúng **1** segment (1-1) → tự động tách bảng `category(category_id, segment)` riêng, rút gọn `product` chỉ giữ khóa ngoại `category_id` (loại phụ thuộc bắc cầu `product_id → category → segment`).
- Nếu **1** category có nhiều segment khác nhau → giữ nguyên `product` với cả 2 cột `category` và `segment` như bản gốc, không ép tách.

In [6]:
df_prod_raw = load("product")
prod_cols = ["product_id", "product_name", "category", "segment", "size", "color", "price", "cogs"]
prod_cols = [c for c in prod_cols if c in df_prod_raw.columns]
product_flat = df_prod_raw[prod_cols].drop_duplicates(subset="product_id").reset_index(drop=True)

# --- Kiểm tra: category -> segment có phải phụ thuộc hàm 1-1 không? ---
check_prod = product_flat.groupby("category")["segment"].nunique()
n_violation_prod = (check_prod > 1).sum()
SPLIT_PRODUCT = (n_violation_prod == 0)
print(f"Số category ứng với NHIỀU hơn 1 segment: {n_violation_prod}")
print("->", "Xác nhận 1-1: TÁCH bảng CATEGORY riêng."
      if SPLIT_PRODUCT else "KHÔNG phải 1-1: GIỮ NGUYÊN PRODUCT với cả category và segment.")

if SPLIT_PRODUCT:
    category = (product_flat[["category", "segment"]].drop_duplicates()
                            .sort_values("category").reset_index(drop=True))
    category.insert(0, "category_id", category.index + 1)

    product = product_flat.merge(category[["category_id", "category"]], on="category")
    product = product.drop(columns=["category", "segment"])
    ordered = ["product_id", "product_name", "category_id", "size", "color", "price", "cogs"]
    product = product[[c for c in ordered if c in product.columns]]

    export(category, "category")
    export(product, "product")
else:
    category = None
    product = product_flat.copy()
    export(product, "product")

product.head()

Số category ứng với NHIỀU hơn 1 segment: 3
-> KHÔNG phải 1-1: GIỮ NGUYÊN PRODUCT với cả category và segment.
Đã xuất product.csv — 2412 dòng, 8 cột -> warehouse_3nf\product.csv


,product_id,product_name,category,segment,size,color,price,cogs
0,1,DragonWear MA-01,Casual,All-weather,M,black,4945.500000,2732.883300
1,2,DragonWear MA-02,Casual,All-weather,L,orange,39.062877,22.660233
2,3,DragonWear MA-03,Casual,All-weather,XL,blue,10831.377188,10289.808329
3,4,DragonWear MA-04,Casual,All-weather,S,white,9610.756522,5604.032128
4,5,DragonWear MA-05,Casual,All-weather,M,purple,8946.000000,8498.700000


## Bước 5/15: PROMOTION *(promo_id)*
Không đổi cấu trúc so với Silver (đã merge `eprom.json` + `promotions.csv`).

In [7]:
df_promo_raw = load("promotion")

promo_cols = ["promo_id", "promo_name", "promo_type", "discount_value", "start_date", "end_date",
              "applicable_category", "promo_channel", "stackable_flag", "min_order_value"]
promo_cols = [c for c in promo_cols if c in df_promo_raw.columns]
promotion = df_promo_raw[promo_cols].drop_duplicates(subset="promo_id").reset_index(drop=True)
export(promotion, "promotion")
promotion.head()

Đã xuất promotion.csv — 50 dòng, 10 cột -> warehouse_3nf\promotion.csv


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,all categories,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,all categories,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,all categories,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,all categories,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


## Bước 6/15: ORDER *(order_id)*
Bỏ `city`, `region`, `district` (dư thừa qua `zip`), bỏ `category`, `segment` (dư thừa qua ORDER_ITEMS → PRODUCT),
bỏ `payment_method` (chuyển hẳn về PAYMENT — không giữ trùng nữa).

In [8]:
df_order_raw = load("order")

order_cols = ["order_id", "order_date", "customer_id", "zip", "order_status",
              "device_type", "order_source", "sales_employee_id"]
order_cols = [c for c in order_cols if c in df_order_raw.columns]
order = df_order_raw[order_cols].drop_duplicates(subset="order_id").reset_index(drop=True)
export(order, "order")
order.head()

Đã xuất order.csv — 646945 dòng, 8 cột -> warehouse_3nf\order.csv


,order_id,order_date,customer_id,zip,order_status,device_type,order_source,sales_employee_id
0,1,2012-07-04,58578,1109,delivered,desktop,paid_search,EMP0103
1,2,2012-07-04,58621,1330,returned,mobile,paid_search,EMP0180
2,3,2012-07-04,58811,1473,delivered,desktop,direct,EMP0093
3,4,2012-07-04,59453,2360,delivered,desktop,referral,EMP0015
4,6,2012-07-06,57821,2886,delivered,mobile,email_campaign,EMP0107


## Bước 7-8/15: ORDER_ITEMS *(order_id, product_id)* + ORDER_ITEM_PROMOTION *(order_id, product_id, promo_id)*
`order_items_silver.csv` đang chứa `promo_id` và `promo_id_2` (nhóm lặp — vi phạm 1NF).
Cell dưới **tách 2 cột này thành các dòng riêng** trong bảng `ORDER_ITEM_PROMOTION`, để 1 dòng hàng có thể
áp 0, 1, 2 hoặc nhiều khuyến mãi mà không cần thêm cột.

In [9]:
df_oi_raw = load("order_items")

# ---- 7. ORDER_ITEMS: chỉ giữ các cột thuộc về chính dòng hàng ----
oi_cols = ["order_id", "product_id", "quantity", "unit_price", "discount_amount"]
oi_cols = [c for c in oi_cols if c in df_oi_raw.columns]
order_items = df_oi_raw[oi_cols].drop_duplicates(subset=["order_id", "product_id"]).reset_index(drop=True)
export(order_items, "order_items")

# ---- 8. ORDER_ITEM_PROMOTION: unpivot promo_id + promo_id_2 thành nhiều dòng ----
df_oip_raw = load("order_item_promotion")
oip_cols = ["order_id", "product_id", "promo_id"]
oip_cols = [c for c in oip_cols if c in df_oip_raw.columns]
order_item_promotion = df_oip_raw[oip_cols].drop_duplicates().reset_index(drop=True)
export(order_item_promotion, "order_item_promotion")

order_items.head()

Đã xuất order_items.csv — 714653 dòng, 5 cột -> warehouse_3nf\order_items.csv
Đã xuất order_item_promotion.csv — 276309 dòng, 3 cột -> warehouse_3nf\order_item_promotion.csv


,order_id,product_id,quantity,unit_price,discount_amount
0,1,2400,7,1138.22,0.0
1,2,609,7,10166.25,0.0
2,3,396,3,11220.33,0.0
3,4,635,5,10639.25,0.0
4,6,1935,1,1597.84,0.0


In [10]:
order_item_promotion.head()

,order_id,product_id,promo_id
0,46253,2250,PROMO-0006
1,46254,2251,PROMO-0006
2,46257,785,PROMO-0006
3,46257,786,PROMO-0006
4,46258,1093,PROMO-0006


## Bước 9/15: SHIPPER *(shipper_id)*
File `shipments_silver.csv` gốc chứa cả thông tin shipper (tên, tuổi, xe...) lẫn `city/region/district`
nhưng **không có sẵn `zip`** — cần ánh xạ ngược qua bảng GEOGRAPHY (khớp theo `city+region+district`).
⚠️ Nếu tổ hợp `(city, region, district)` không map ra đúng **1** zip duy nhất (VD: 1 quận có nhiều mã zip),
cell dưới sẽ in cảnh báo — cần nhóm xác nhận lại dữ liệu thật, không tự ý chọn đại 1 zip.

In [11]:
print(load("shipper").columns.tolist())

['shipper_id', 'shipper_name', 'shipper_phone', 'shipper_gender', 'shipper_age', 'shipper_marital_status', 'shipper_education', 'shipper_company', 'shipper_vehicle', 'shipper_experience_years', 'shipper_rating', 'delivery_success_rate', 'average_delivery_time', 'working_shift', 'join_date', 'zip']


In [12]:
df_ship_person_raw = load("shipper")
shipper_cols = ["shipper_id", "shipper_name", "shipper_phone", "shipper_gender", "shipper_age",
                 "shipper_marital_status", "shipper_education", "shipper_company", "shipper_vehicle",
                 "shipper_experience_years", "shipper_rating", "delivery_success_rate",
                 "average_delivery_time", "working_shift", "join_date", "zip"]
shipper_cols = [c for c in shipper_cols if c in df_ship_person_raw.columns]
shipper = df_ship_person_raw[shipper_cols].drop_duplicates(subset="shipper_id").reset_index(drop=True)
export(shipper, "shipper")
shipper.head()

Đã xuất shipper.csv — 80 dòng, 16 cột -> warehouse_3nf\shipper.csv


,shipper_id,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,zip
0,SHP00001,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Viettel Post,Truck,7,5.0,99.0,61,Evening,2026-03-17,38006
1,SHP00002,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,J&T Express,Van,2,4.9,98.4,72,Afternoon,2025-01-29,52231
2,SHP00003,Hoàng Thị Khánh,927142576,Male,30,Single,High School,GHN,Motorbike,10,4.8,95.1,53,Evening,2019-11-13,3
3,SHP00004,Trần Đức Vy,971617475,Female,42,Married,College,Viettel Post,Truck,8,5.0,96.3,53,Evening,2025-12-22,50611
4,SHP00005,Trần Minh Cường,979196342,Male,31,Married,College,BEST Express,Truck,10,4.6,95.7,62,Morning,2019-12-19,75102


## Bước 10/15: SHIPMENT *(order_id, shipper_id)*
Chỉ giữ các cột thuộc về **sự kiện giao hàng**, tách khỏi thông tin cá nhân shipper (đã tách ở Bước 9).

In [13]:
df_shipment_raw = load("shipment")
shipment_cols = ["order_id", "shipper_id", "ship_date", "delivery_date", "shipping_fee"]
shipment_cols = [c for c in shipment_cols if c in df_shipment_raw.columns]
shipment = df_shipment_raw[shipment_cols].drop_duplicates(subset=["order_id", "shipper_id"]).reset_index(drop=True)
export(shipment, "shipment")
shipment.head()

Đã xuất shipment.csv — 59559 dòng, 5 cột -> warehouse_3nf\shipment.csv


,order_id,shipper_id,ship_date,delivery_date,shipping_fee
0,1,SHP00001,2012-07-07,2012-07-11,1.37
1,2,SHP00002,2012-07-06,2012-07-10,2.60
2,3,SHP00003,2012-07-04,2012-07-07,2.38
3,4,SHP00004,2012-07-05,2012-07-11,2.49
4,6,SHP00005,2012-07-09,2012-07-16,25.79


## Bước 11/15: PAYMENT *(order_id)*
Không đổi cấu trúc so với Silver.

In [14]:
df_pay_raw = load("payment")
pay_cols = ["order_id", "payment_method", "payment_value", "installments"]
pay_cols = [c for c in pay_cols if c in df_pay_raw.columns]
payment = df_pay_raw[pay_cols].drop_duplicates(subset="order_id").reset_index(drop=True)
export(payment, "payment")
payment.head()

Đã xuất payment.csv — 646945 dòng, 4 cột -> warehouse_3nf\payment.csv


,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


## Bước 12/15: RETURNS *(return_id)*
Không đổi cấu trúc so với Silver — lưu ý FK `(order_id, product_id)` giờ tham chiếu tới composite key của ORDER_ITEMS.

In [15]:
df_ret_raw = load("returns")
ret_cols = ["return_id", "order_id", "product_id", "return_date", "return_reason",
            "return_quantity", "refund_amount"]
ret_cols = [c for c in ret_cols if c in df_ret_raw.columns]
returns = df_ret_raw[ret_cols].drop_duplicates(subset="return_id").reset_index(drop=True)
export(returns, "returns")
returns.head()

Đã xuất returns.csv — 35733 dòng, 7 cột -> warehouse_3nf\returns.csv


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,1,2400,2012-07-11,wrong_size,1,1138.22
1,RET-000002,24,671,2012-07-12,defective,3,32590.56
2,RET-000003,47,1449,2012-07-13,changed_mind,4,7038.36
3,RET-000004,72,1942,2012-07-14,not_as_described,5,7295.85
4,RET-000005,93,485,2012-07-15,late_delivery,3,29812.71


## Bước 13/15: REVIEWS *(review_id)*
Không đổi cấu trúc so với Silver — giữ `customer_id` độc lập (người review không nhất thiết là người đặt đơn).

In [16]:
df_rev_raw = load("reviews")
rev_cols = ["review_id", "order_id", "product_id", "customer_id", "review_date", "rating", "review_title"]
rev_cols = [c for c in rev_cols if c in df_rev_raw.columns]
reviews = df_rev_raw[rev_cols].drop_duplicates(subset="review_id").reset_index(drop=True)
export(reviews, "reviews")
reviews.head()

Đã xuất reviews.csv — 714653 dòng, 6 cột -> warehouse_3nf\reviews.csv


,review_id,order_id,product_id,review_date,rating,review_title
0,REV-0000001,1,2400,2012-07-10,5,Highly recommend
1,REV-0000002,2,609,2012-07-11,5,Very satisfied
2,REV-0000003,3,396,2012-07-12,5,Excellent product!
3,REV-0000004,4,635,2012-07-13,5,Great quality
4,REV-0000005,6,1935,2012-07-14,5,Highly recommend


## Bước 14/15: INVENTORY *(snapshot_date, product_id)*
Bỏ `product_name`, `category`, `segment` (phụ thuộc bộ phận vào `product_id` — vi phạm 2NF),
bỏ `year`, `month` (phụ thuộc bộ phận vào `snapshot_date`, tính lại bằng hàm ngày khi cần).

In [17]:
df_inv_raw = load("inventory")
inv_cols = ["snapshot_date", "product_id", "stock_on_hand", "units_received", "units_sold",
            "stockout_days", "days_of_supply", "fill_rate", "stockout_flag", "overstock_flag",
            "reorder_flag", "sell_through_rate"]
inv_cols = [c for c in inv_cols if c in df_inv_raw.columns]
inventory = df_inv_raw[inv_cols].drop_duplicates(subset=["snapshot_date", "product_id"]).reset_index(drop=True)
export(inventory, "inventory")
inventory.head()

Đã xuất inventory.csv — 60247 dòng, 12 cột -> warehouse_3nf\inventory.csv


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate
0,2022-10-31,1,3,1,1,2,90.0,0.9333,1,0,0,0.2500
1,2022-11-30,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500
2,2022-12-31,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500
3,2016-04-30,3,35,13,11,2,95.5,0.9333,1,1,0,0.2391
4,2016-05-31,3,36,11,10,1,108.0,0.9667,1,1,0,0.2174


## Bước 15/15: WEB_TRAFFIC *(date)*
Không đổi cấu trúc so với Silver.

In [18]:
df_web_raw = load("web_traffic")
web_cols = ["date", "sessions", "unique_visitors", "page_views", "bounce_rate",
            "avg_session_duration_sec", "traffic_source"]
web_cols = [c for c in web_cols if c in df_web_raw.columns]
web_traffic = df_web_raw[web_cols].drop_duplicates(subset="date").reset_index(drop=True)
export(web_traffic, "web_traffic")
web_traffic.head()

Đã xuất web_traffic.csv — 3652 dòng, 7 cột -> warehouse_3nf\web_traffic.csv


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


## Bước 16: Tóm tắt quyết định tách bảng GEOGRAPHY & PRODUCT
Hai kiểm tra phụ thuộc hàm đã được thực hiện **trực tiếp ở Bước 1 và Bước 4** (không cần chạy lại ở đây).
Cell dưới chỉ tổng hợp lại quyết định mà notebook đã tự đưa ra dựa trên dữ liệu thật.

In [19]:
print("=== GEOGRAPHY ===")
print("Đã tách REGION/CITY/DISTRICT/GEOGRAPHY:", SPLIT_GEOGRAPHY)
if SPLIT_GEOGRAPHY:
    print(f"  region  : {region.shape[0]} dòng")
    print(f"  city    : {city.shape[0]} dòng")
    print(f"  district: {district.shape[0]} dòng")
print(f"  geography: {geography.shape[0]} dòng, {geography.shape[1]} cột -> {list(geography.columns)}")

print("\n=== PRODUCT ===")
print("Đã tách CATEGORY riêng:", SPLIT_PRODUCT)
if SPLIT_PRODUCT:
    print(f"  category: {category.shape[0]} dòng")
print(f"  product : {product.shape[0]} dòng, {product.shape[1]} cột -> {list(product.columns)}")

=== GEOGRAPHY ===
Đã tách REGION/CITY/DISTRICT/GEOGRAPHY: False
  geography: 39948 dòng, 4 cột -> ['zip', 'city', 'region', 'district']

=== PRODUCT ===
Đã tách CATEGORY riêng: False
  product : 2412 dòng, 8 cột -> ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']


## Tổng kết: xác nhận đủ 15 file đã xuất
Liệt kê lại toàn bộ file trong `OUTPUT_DIR` kèm số dòng, đối chiếu với danh sách 15 bảng trong
`Luoc_do_quan_he_3NF.docx`.

In [20]:
expected_tables = [
    "geography", "shipper", "customer", "sales_employee", "product", "promotion",
    "order", "order_items", "order_item_promotion", "shipment", "payment",
    "returns", "reviews", "inventory", "web_traffic",
]
if SPLIT_GEOGRAPHY:
    expected_tables += ["region", "city", "district"]
if SPLIT_PRODUCT:
    expected_tables += ["category"]

print(f"{'Bảng':<25}{'Trạng thái':<15}{'Số dòng'}")
print("-" * 55)
for t in expected_tables:
    fp = os.path.join(OUTPUT_DIR, f"{t}.csv")
    if os.path.exists(fp):
        n = sum(1 for _ in open(fp, encoding="utf-8-sig")) - 1
        print(f"{t:<25}{'OK':<15}{n}")
    else:
        print(f"{t:<25}{'THIẾU':<15}-")

print(f"\nTổng số bảng OLTP 3NF đã xuất: {len(expected_tables)}")

Bảng                     Trạng thái     Số dòng
-------------------------------------------------------
geography                OK             39948
shipper                  OK             80
customer                 OK             121930
sales_employee           OK             200
product                  OK             2412
promotion                OK             50
order                    OK             646945
order_items              OK             714653
order_item_promotion     OK             276309
shipment                 OK             59559
payment                  OK             646945
returns                  OK             35733
reviews                  OK             714653
inventory                OK             60247
web_traffic              OK             3652

Tổng số bảng OLTP 3NF đã xuất: 15


---
## Bước 17: Xây dựng và xuất bảng FACT/DIM cho Data Warehouse (Đề tài 4)

Từ các bảng 3NF vừa xuất ở trên, xây dựng mô hình **star schema** phục vụ Đề tài 4 (Đánh giá năng lực nhân viên bán hàng):

- **Fact_Sales**: mỗi dòng = 1 dòng hàng trong `ORDER_ITEMS`, gắn với `ORDER` (ngày, khách hàng, nhân viên, zip, trạng thái) và số khuyến mãi áp dụng từ `ORDER_ITEM_PROMOTION`.
- **Dim_Date**: lịch đầy đủ từ ngày đơn hàng sớm nhất đến muộn nhất.
- **Dim_Customer**: từ `CUSTOMER`, kèm city/region/district (denormalize để dễ lọc/group trên BI).
- **Dim_Employee**: từ `SALES_EMPLOYEE`.
- **Dim_Product**: từ `PRODUCT` (+ `CATEGORY` nếu đã tách ở Bước 4, gộp lại cho dễ dùng trên star schema).
- **Dim_Geography**: từ `GEOGRAPHY` (+ `DISTRICT`/`CITY`/`REGION` nếu đã tách ở Bước 1, gộp lại tương tự).

Kết quả xuất vào thư mục con `star_schema/` bên trong `OUTPUT_DIR`.

In [21]:
# ==== Cấu hình xuất FACT/DIM ====
DW_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "star_schema")
os.makedirs(DW_OUTPUT_DIR, exist_ok=True)

def export_dw(df, table_name):
    """Xuất 1 DataFrame Fact/Dim ra star_schema/<table_name>.csv"""
    out_path = os.path.join(DW_OUTPUT_DIR, f"{table_name}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Đã xuất {table_name}.csv — {df.shape[0]} dòng, {df.shape[1]} cột -> {out_path}")
    return df

# order_date cần ở dạng datetime để dùng cho Dim_Date và Fact_Sales
order["order_date"] = pd.to_datetime(order["order_date"])

### 17.1 Dim_Date

In [22]:
min_date = order["order_date"].min()
max_date = order["order_date"].max()
all_dates = pd.date_range(min_date, max_date, freq="D")

dim_date = pd.DataFrame({"date_key": all_dates})
dim_date["year"] = dim_date["date_key"].dt.year
dim_date["quarter"] = dim_date["date_key"].dt.quarter
dim_date["month"] = dim_date["date_key"].dt.month
dim_date["month_name"] = dim_date["date_key"].dt.strftime("%B")
dim_date["day"] = dim_date["date_key"].dt.day
dim_date["day_of_week"] = dim_date["date_key"].dt.dayofweek  # 0 = Thứ Hai
dim_date["day_name"] = dim_date["date_key"].dt.strftime("%A")
dim_date["is_weekend"] = dim_date["day_of_week"].isin([5, 6])
dim_date["year_month"] = dim_date["date_key"].dt.strftime("%Y-%m")

export_dw(dim_date, "Dim_Date")
dim_date.head()

Đã xuất Dim_Date.csv — 3833 dòng, 10 cột -> warehouse_3nf\star_schema\Dim_Date.csv


,date_key,year,quarter,month,month_name,day,day_of_week,day_name,is_weekend,year_month
0,2012-07-04,2012,3,7,July,4,2,Wednesday,False,2012-07
1,2012-07-05,2012,3,7,July,5,3,Thursday,False,2012-07
2,2012-07-06,2012,3,7,July,6,4,Friday,False,2012-07
3,2012-07-07,2012,3,7,July,7,5,Saturday,True,2012-07
4,2012-07-08,2012,3,7,July,8,6,Sunday,True,2012-07


### 17.2 Dim_Customer

In [23]:
dim_customer = customer.merge(geo_flat[["zip", "city", "region", "district"]], on="zip", how="left")
export_dw(dim_customer, "Dim_Customer")
dim_customer.head()

Đã xuất Dim_Customer.csv — 121930 dòng, 9 cột -> warehouse_3nf\star_schema\Dim_Customer.csv


,customer_id,zip,signup_date,gender,age_group,acquisition_channel,city,region,district
0,1,15201,2021-12-30,Female,35-44,social_media,Hai Phong,East,District #13
1,2,15201,2013-12-27,Female,45-54,email_campaign,Hai Phong,East,District #13
2,3,15201,2018-07-24,Female,18-24,organic_search,Hai Phong,East,District #13
3,4,15201,2017-11-29,Male,35-44,referral,Hai Phong,East,District #13
4,5,15201,2022-09-23,Male,55+,organic_search,Hai Phong,East,District #13


### 17.3 Dim_Employee

In [24]:
dim_employee = sales_employee.copy()
export_dw(dim_employee, "Dim_Employee")
dim_employee.head()

Đã xuất Dim_Employee.csv — 200 dòng, 2 cột -> warehouse_3nf\star_schema\Dim_Employee.csv


,sales_employee_id,name
0,EMP0103,Nguyễn Thị Phúc
1,EMP0180,Lê Gia Hùng
2,EMP0093,Bùi Minh Quân
3,EMP0015,Võ Quốc Mai
4,EMP0107,Hồ Thanh Yến


### 17.4 Dim_Product

In [25]:
if SPLIT_PRODUCT:
    dim_product = product.merge(category, on="category_id", how="left").drop(columns="category_id")
    dim_product = dim_product[["product_id", "product_name", "category", "segment", "size", "color", "price", "cogs"]]
else:
    dim_product = product.copy()

export_dw(dim_product, "Dim_Product")
dim_product.head()

Đã xuất Dim_Product.csv — 2412 dòng, 8 cột -> warehouse_3nf\star_schema\Dim_Product.csv


,product_id,product_name,category,segment,size,color,price,cogs
0,1,DragonWear MA-01,Casual,All-weather,M,black,4945.500000,2732.883300
1,2,DragonWear MA-02,Casual,All-weather,L,orange,39.062877,22.660233
2,3,DragonWear MA-03,Casual,All-weather,XL,blue,10831.377188,10289.808329
3,4,DragonWear MA-04,Casual,All-weather,S,white,9610.756522,5604.032128
4,5,DragonWear MA-05,Casual,All-weather,M,purple,8946.000000,8498.700000


### 17.5 Dim_Geography

In [26]:
if SPLIT_GEOGRAPHY:
    dim_geography = (geography.merge(district, on="district_id")
                               .merge(city, on="city_id")
                               .merge(region, on="region_id")
                               [["zip", "district", "city", "region"]])
else:
    dim_geography = geography.copy()

export_dw(dim_geography, "Dim_Geography")
dim_geography.head()

Đã xuất Dim_Geography.csv — 39948 dòng, 4 cột -> warehouse_3nf\star_schema\Dim_Geography.csv


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


### 17.6 Fact_Sales

In [27]:
# Doanh thu gộp / ròng ở mức dòng hàng (grain = 1 dòng ORDER_ITEMS)
oi = order_items.copy()
oi["gross_amount"] = oi["quantity"] * oi["unit_price"]
oi["net_amount"] = oi["gross_amount"] - oi["discount_amount"]

# Số khuyến mãi áp dụng cho từng dòng hàng (từ ORDER_ITEM_PROMOTION)
promo_count = (order_item_promotion.groupby(["order_id", "product_id"])
                                    .size().rename("promo_count").reset_index())

fact_sales = oi.merge(promo_count, on=["order_id", "product_id"], how="left")
fact_sales["promo_count"] = fact_sales["promo_count"].fillna(0).astype(int)
fact_sales["has_promotion"] = fact_sales["promo_count"] > 0

# Gắn thông tin đơn hàng: ngày, khách hàng, nhân viên, zip, trạng thái
fact_sales = fact_sales.merge(
    order[["order_id", "order_date", "customer_id", "zip", "sales_employee_id", "order_status"]],
    on="order_id", how="inner"
).rename(columns={"order_date": "date_key"})

fact_cols = ["order_id", "product_id", "date_key", "customer_id", "sales_employee_id", "zip",
             "order_status", "quantity", "unit_price", "discount_amount",
             "gross_amount", "net_amount", "promo_count", "has_promotion"]
fact_sales = fact_sales[fact_cols]

export_dw(fact_sales, "Fact_Sales")
fact_sales.head()

Đã xuất Fact_Sales.csv — 714653 dòng, 14 cột -> warehouse_3nf\star_schema\Fact_Sales.csv


,order_id,product_id,date_key,customer_id,sales_employee_id,zip,order_status,quantity,unit_price,discount_amount,gross_amount,net_amount,promo_count,has_promotion
0,1,2400,2012-07-04,58578,EMP0103,1109,delivered,7,1138.22,0.0,7967.54,7967.54,0,False
1,2,609,2012-07-04,58621,EMP0180,1330,returned,7,10166.25,0.0,71163.75,71163.75,0,False
2,3,396,2012-07-04,58811,EMP0093,1473,delivered,3,11220.33,0.0,33660.99,33660.99,0,False
3,4,635,2012-07-04,59453,EMP0015,2360,delivered,5,10639.25,0.0,53196.25,53196.25,0,False
4,6,1935,2012-07-06,57821,EMP0107,2886,delivered,1,1597.84,0.0,1597.84,1597.84,0,False


### 17.7 Kiểm tra chéo khóa ngoại Fact_Sales với các Dim

In [28]:
print("Số dòng Fact_Sales:", f"{len(fact_sales):,}")
print("Tổng net_amount   :", f"{fact_sales['net_amount'].sum():,.2f}")

missing_cust = (~fact_sales["customer_id"].isin(dim_customer["customer_id"])).sum()
missing_emp  = (~fact_sales["sales_employee_id"].isin(dim_employee["sales_employee_id"])).sum()
missing_prod = (~fact_sales["product_id"].isin(dim_product["product_id"])).sum()
missing_zip  = (~fact_sales["zip"].isin(dim_geography["zip"])).sum()
missing_date = (~fact_sales["date_key"].isin(dim_date["date_key"])).sum()

print("\nSố dòng Fact_Sales có FK KHÔNG khớp Dim tương ứng (kỳ vọng = 0):")
print(f"  -> Dim_Customer  : {missing_cust}")
print(f"  -> Dim_Employee  : {missing_emp}")
print(f"  -> Dim_Product   : {missing_prod}")
print(f"  -> Dim_Geography : {missing_zip}")
print(f"  -> Dim_Date      : {missing_date}")

if max(missing_cust, missing_emp, missing_prod, missing_zip, missing_date) == 0:
    print("\nOK: toàn bộ khóa ngoại trong Fact_Sales đều khớp với các bảng Dim.")
else:
    print("\nCẢNH BÁO: có FK không khớp — kiểm tra lại dữ liệu nguồn trước khi đưa vào BI.")

Số dòng Fact_Sales: 714,653
Tổng net_amount   : 15,905,743,139.85

Số dòng Fact_Sales có FK KHÔNG khớp Dim tương ứng (kỳ vọng = 0):
  -> Dim_Customer  : 0
  -> Dim_Employee  : 0
  -> Dim_Product   : 0
  -> Dim_Geography : 0
  -> Dim_Date      : 0

OK: toàn bộ khóa ngoại trong Fact_Sales đều khớp với các bảng Dim.


### 17.8 Tổng kết các bảng star schema đã xuất

In [29]:
star_tables = ["Dim_Date", "Dim_Customer", "Dim_Employee", "Dim_Product", "Dim_Geography", "Fact_Sales"]

print(f"{'Bảng':<18}{'Trạng thái':<15}{'Số dòng'}")
print("-" * 45)
for t in star_tables:
    fp = os.path.join(DW_OUTPUT_DIR, f"{t}.csv")
    if os.path.exists(fp):
        n = sum(1 for _ in open(fp, encoding="utf-8-sig")) - 1
        print(f"{t:<18}{'OK':<15}{n}")
    else:
        print(f"{t:<18}{'THIẾU':<15}-")

Bảng              Trạng thái     Số dòng
---------------------------------------------
Dim_Date          OK             3833
Dim_Customer      OK             121930
Dim_Employee      OK             200
Dim_Product       OK             2412
Dim_Geography     OK             39948
Fact_Sales        OK             714653
